## Description
Main python script used to assess climate characteristics and adaptation measures in Pisor, Touma et al. (in prep)<br>
Sricpts by [Danielle Touma](danielletouma.com)<br>
Run script in Jupyter Notebook or platform with *.ipynb capabilities.

### External functions
[find_runs.py](https://gist.github.com/alimanfoo/c5977e87111abe8127453b21204c1065)

## Main script

### Import packages/libraries

In [ ]:
# analysis packages
import numpy as np
import pandas as pd
from scipy.ndimage import label, binary_dilation, generate_binary_structure
from find_runs import * # function found in find_runs.py
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt

### Basic set up

Initialize directories where you can find the precipitation dataset (dir_month),
the threshold dataset (dir_pxx), shapefile (dir_shp), and remittance data (dir_rem), and
where figures should be output (dir_fig).

In [ ]:
dir_spei = "/disk/dtouma/CEDA/"
dir_ndvi = "/disk/dtouma/NDVI/Africa/monthly_average/"
dir_fig = "/disk/dtouma/figures/"
dir_shp =  "/disk/dtouma/shapefiles/"
dir_rem = "/disk/dtouma/remittance/"
dir_lights =  '/disk/dtouma/nightlights/'

### Read in SPEI files

We use CHIRPS-hPET SPEI Data.

More info:

Website for download: https://catalogue.ceda.ac.uk/uuid/ac43da11867243a1bb414e1637802dec
paper: https://essd.copernicus.org/articles/15/5449/2023/

Available datasets: 
SPEI from 1 to 48 months. 1981-2022. Global.

Precipitation datasets:
- CHIRPS (0.05 degrees)
- MSWEP (0.25 degrees)
- CRU-TS (0.5 degrees)

PET datasts:
- GLEAM (Priestley–Taylor equation; satellite & reanalysis data; 0.25 degrees)
- hPET (Penman Monteith equation; ERA5-Land reanalysis; 0.1 degrees)
- CRU-TS (Penman Monteith equation; CRU-TS; 0.5 degrees)

SPEI datasets:
CHIRPS_GLEAM, CHIRPS_hPET, MSWEP_GLEAM, MSWEP_hPET. All are statistically interpolated to 0.05 degrees.

**We use the CHIRPS_GLEAM dataset at 0.05 resolution, already subset for Africa, for 1-, 3-, 6- and 12-month SPEI.**

In [ ]:
# Reading subsetted Africa SPEI for different time scales:
# 1-, 3-, 6- and 12-month SPEI.
# This will allow us to better understand different timescales of drought

spei_time_scales = [1, 3, 6, 12]
spei_file_list = []
for tt in spei_time_scales:
    spei_file_list.append(dir_spei+'Africa_spei'+"%02d" % (tt,)+'.nc')

spei_all_xr = xr.open_mfdataset(spei_file_list, combine='nested' ,concat_dim='time_scale')

In [ ]:
# subsetting for time period until survey year since that is what people feel "climatologically"
survey_year = 2009
spei_all_years = spei_all_xr.time.dt.year
spei_sub_xr = spei_all_xr.sel(time=spei_all_xr.time.dt.year.isin(np.arange(spei_all_years[0],survey_year+1,1)))

# convert xarray into numpy array
spei_sub_np = np.array(spei_sub_xr.spei)

Initialize variables that describe the dimensions of your SPEI dataset.

In [ ]:
time_xr = spei_sub_xr.time
lon_xr = spei_sub_xr.lon 
lat_xr = spei_sub_xr.lat 

nlat = len(lat_xr)
nlon = len(lon_xr)
ntime = len(time_xr)

lon = np.array(lon_xr)
lat = np.array(lat_xr)

Calculate grid area for dataset

In [ ]:
lat_sep = lat[1]-lat[0]
lon_sep = lon[1]-lon[0]

def gridsize(lat1, lon1, lat2, lon2):
   #https://en.wikipedia.org/wiki/Haversine_formula
   #https://stackoverflow.com/questions/639695/how-to-convert-latitude-or-longitude-to-meters/11172685#11172685
   R = 6378.137 # // Radius of earth in km
   dLat = lat2 * np.pi / 180 - lat1 * np.pi / 180
   dLon = lon2 * np.pi / 180 - lon1 * np.pi / 180
   a = np.sin(dLat/2) * np.sin(dLat/2) + np.cos(lat1 * np.pi / 180) * np.cos(lat2 * np.pi / 180) * np.sin(dLon/2) * np.sin(dLon/2)
   c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
   d = R * c
   return d * 1000 #; // meters

grid_area = np.zeros(shape=(nlat,nlon))

for ii in range(0,nlat,1):
   for jj in range(0,nlon,1):
      grid_area[ii,jj] = gridsize((lat[ii]-(lat_sep/2)),(lon[jj]-(lon_sep/2)),
                                  (lat[ii]+(lat_sep/2)),(lon[jj]+(lon_sep/2)))

### Read in NDVI monthly average file<br>
NDVI monthly data for Africa has been processed from global daily files (subset spatially and temporally averaged). <br>
Original netCDF files are downloaded from NOAA Climate Data Record (CDR) of AVHRR Normalized Difference Vegetation Index (NDVI), Version 5 (https://www.ncei.noaa.gov/metadata/geoportal/rest/metadata/item/gov.noaa.ncdc:C01558/html). <br> Processing was done in CDO - a command line language that can be used to process netCDF files. <br> The average monthly file that was used in this study can be found on the GitHub.

In [ ]:
year0 = 1982
year1 = 2009

ndvi_xr = xr.open_dataset(dir_ndvi+'AVHRR-Land_v005_AVH13C1_NOAA_'+str(year0)+'-'+str(year1)+'_monthly_average_Africa.nc')    

Calculate annual mean NDVI

In [ ]:
ndvi_annual_mean = ndvi_xr.NDVI.groupby('time.year').mean('time')

Calculate SVI (Standardized vegetation index)<br>
$SVI = (NDVI_{t,i,j} - \mu_{i,j})/ \sigma_{i,j}$<br>
The SVI reflects if the vegetation is relatively healthy (>0; wet conditions) or relatively unhealthy (<0; drought conditions).

In [ ]:
# find monthly mean and standard deviation across all dataset
ndvi_mu_month = ndvi_xr.NDVI.groupby('time.month').mean('time')
ndvi_sig_month = ndvi_xr.NDVI.groupby('time.month').std('time')
# standardize using (NDVI - mu)/sigma
SVI = (ndvi_xr.NDVI - ndvi_mu_month.sel(month=ndvi_xr.NDVI.time.dt.month))/ndvi_sig_month.sel(month=ndvi_xr.NDVI.time.dt.month)

### Regrid SVI and NDVI to SPEI grid

Build regridder

In [ ]:
ds_out = xr.Dataset(
    {
        "lat": spei_sub_xr.lat,
        "lon": spei_sub_xr.lon,
    }
)

regridder = xe.Regridder(SVI, ds_out, "bilinear")


regrid SVI, NDVI, and NDVI annual mean using regridder

In [ ]:
SVI_regridded = regridder(SVI)
NDVI_regridded = regridder(ndvi_xr.NDVI)
NDVI_annual_mean_regridded = regridder(ndvi_annual_mean)

In [ ]:
SVI_regridded_np = np.array(SVI_regridded)
NDVI_regridded_np = np.array(NDVI_regridded)
NDVI_annual_regridded_np = np.array(NDVI_annual_mean_regridded) 

Pad the arrays so that the time dimension is the same as the SPEI array. In this case, the NDVI data starts one year after the SPEI, so we have a whole year of missing SVI values in 1981. 

In [ ]:
SVI_regridded_np_padded = np.pad(SVI_regridded_np, [(12, 0), (0, 0), (0, 0)], mode='constant', constant_values=-999999)
SVI_regridded_np_padded = np.ma.masked_where(SVI_regridded_np_padded==-999999, SVI_regridded_np_padded)
NDVI_regridded_np_padded = np.pad(NDVI_regridded_np, [(12, 0), (0, 0), (0, 0)], mode='constant', constant_values=-999999)
NDVI_regridded_np_padded = np.ma.masked_where(NDVI_regridded_np_padded==-999999, NDVI_regridded_np_padded)
NDVI_annual_regridded_np_padded = np.pad(NDVI_annual_regridded_np, [(12, 0), (0, 0), (0, 0)], mode='constant', constant_values=-999999)
NDVI_annual_regridded_np_padded = np.ma.masked_where(NDVI_annual_regridded_np_padded==-999999, NDVI_annual_regridded_np_padded)

#### Finding droughts and calculating frequency, DIMS, and spatial extent.

Create a binary variable storing whether or not a pixel is below a certain drought threshold (1 = drought, 0 = no drought) for several percentile thresholds.
First calculate percentile threshold values and then find the locations and times when the spei value is equal to or less than the percentile threshold.

In [ ]:
# calculate percentile threshold values
percentiles = np.array([2, 5, 10, 20, 50]) # severe drought to moderately dry conditions
spei_percentiles = np.zeros((len(spei_time_scales),len(percentiles),nlat,nlon))
for tt in range(0,len(spei_time_scales),1):
    print(str(spei_time_scales[tt])+'-month SPEI')
    spei_percentiles[tt,:,:,:] = np.nanpercentile(spei_sub_np[tt,:,:,:], percentiles, axis=0) #spei percentiles has the thresholds for each percentile, time scale and grid point (timescale x percentile x lat x lon)
    # need nanpercentile since end values are nan for >1-month running average for SPEI


Create SPEI binary dataset based on percentiles

In [ ]:
spei_binary_perc = np.zeros((len(spei_time_scales),
                        len(percentiles),
                        ntime,
                        nlat,
                        nlon,),
                        dtype=int)

for mm in range(0,ntime,1):
    if (mm%20==0):
        print(str(mm+1)+' out of '+ str(ntime))
    for pp in range(0,len(percentiles),1):
        percentile_inds = np.where(spei_sub_np[:,mm,:,:]<=spei_percentiles[:,pp,:,:])
        spei_binary_perc[percentile_inds[0],pp,mm,percentile_inds[1],percentile_inds[2]] = 1 


Find one or more consecutive months with drought. drought_onsets = 1 when a drought starts over each grid point. drought_lengths = corresponding drought duration of that drought onset

First, select SPEI time scale and percentile to reduce computational time.

In [ ]:
tt = 3      #1 = 3-month SPEI #3 = 12-month SPEI 
xx = 2      # 10th percentile

In [ ]:
print(str(spei_time_scales[tt])+'-month SPEI, '+str(percentiles[xx])+'th percentile')

# drought onsets and lengths for percentiles
drought_onsets_perc = np.zeros((ntime,nlat,nlon),dtype=float)
drought_lengths_perc = np.zeros((ntime,nlat,nlon),dtype=float)

for ii in range(0,nlat,1): # loop through lats
    for jj in range(0,nlon,1): # loop through lons
        # check if there are any drought months and if the grid point has no rain at all
        if (np.sum(spei_binary_perc[tt,xx,:,ii,jj])>0):
            v, s, l = find_runs(spei_binary_perc[tt,xx,:,ii,jj])
            # v = value of run. in our case, 1 = drought, 0 = no drought
            # s = index of run starts
            s_droughts = s[np.where(v==1)] # find indices of drought starts (i.e., v == 1)
            # l = length of runs
            l_droughts = l[np.where(v==1)] # find lengths of drought runs (i.e., v == 1)
            drought_onsets_perc[s_droughts,ii,jj] = 1
            drought_lengths_perc[s_droughts,ii,jj] = l_droughts
            

For each drought run/event, find the magnitude, severity, and intensity of that drought using the *departure* (*i<sub>1</sub>* and *i<sub>2</sub>* in Figure 1).

In [ ]:
drought_severity_perc = np.full(fill_value = 999.0, shape=(ntime, nlat, nlon) ,dtype=float) 
drought_magnitude_perc = np.full(fill_value = 999.0, shape=(ntime, nlat, nlon) ,dtype=float)
drought_intensity_perc = np.full(fill_value = 999.0, shape=(ntime, nlat, nlon) ,dtype=float)

print(str(spei_time_scales[tt])+'-monthly spei')
print(str(percentiles[xx])+' percentile drought')
for ii in range(0,nlat,1): # loop through lats
    for jj in range(0,nlon,1): # loop through lons
        start_inds = np.where(drought_onsets_perc[:,ii,jj]==1)[0]
        if len(start_inds)>0:
            for ss in start_inds:
                drought_severity_perc[ss,ii,jj] = np.ma.sum(spei_sub_np[tt,ss:ss+int(drought_lengths_perc[ss,ii,jj]),ii,jj])
                drought_magnitude_perc[ss,ii,jj] = np.ma.min(spei_sub_np[tt,ss:ss+int(drought_lengths_perc[ss,ii,jj]),ii,jj])
                drought_intensity_perc[ss,ii,jj] = np.ma.mean(spei_sub_np[tt,ss:ss+int(drought_lengths_perc[ss,ii,jj]),ii,jj])


Find SVI severity during drought events

In [ ]:
# For selected time scale and standard deviation threshold

SVI_drought_severity = np.full(fill_value = 999.0, shape=(ntime, nlat, nlon) ,dtype=float) 

print(str(spei_time_scales[tt])+'-monthly spei')
print(str(percentiles[xx])+' percentile drought')
for ii in range(0,nlat,1): # loop through lats
    for jj in range(0,nlon,1): # loop through lons
        start_inds = np.where(drought_onsets_perc[:,ii,jj]==1)[0]
        if len(start_inds)>0:
            for ss in start_inds:
                SVI_drought_severity[ss,ii,jj] = np.ma.sum(SVI_regridded_np_padded[ss:ss+int(drought_lengths_perc[ss,ii,jj]),ii,jj])


In [ ]:
# Finding spatially connected droughts, or 'drought objects', object area, their total and mean intensity across the object.
# These objects are not connected in time, instead, each month has their own set of objects.

spei_perc_object_ngrids = np.zeros(shape=(ntime,nlat,nlon), dtype='int')
spei_perc_object_area_m2 = np.zeros(shape=(ntime,nlat,nlon)) 
spei_perc_nobjects = np.zeros(ntime, dtype='int')

# user defined structures for finding connected objects
l_structure = [[1,1,1],[1,1,1],[1,1,1]] # structure for finding connected areas - this structure allows edges and vertices to be connected (queen's case) 

print(str(spei_time_scales[tt])+'-month SPEI, '+str(percentiles[xx])+'th percentile')

for mm in range(0,ntime,1):
     mm_binary = spei_binary_perc[tt,xx,mm,:,:]
     mm_labels, mm_nlabels = label(mm_binary, structure=l_structure)
     spei_perc_nobjects[mm] = mm_nlabels
     print(str(mm)+' out of '+str(ntime)+' months: '+str(mm_nlabels))
     for obj in range(0,mm_nlabels,1):
          obj_inds = np.where(mm_labels==(obj+1))
          spei_perc_object_ngrids[mm,obj_inds[0],obj_inds[1]] = len(obj_inds[0])
          spei_perc_object_area_m2[mm,obj_inds[0],obj_inds[1]] = np.sum(grid_area[obj_inds[0],obj_inds[1]])

In [ ]:
print(str(spei_time_scales[tt])+'-month SPEI, '+str(percentiles[xx])+'th percentile')

#mask any places where no drought occured
drought_severity_perc = np.ma.masked_equal(drought_severity_perc,999.0)
drought_magnitude_perc = np.ma.masked_equal(drought_magnitude_perc,999.0)
drought_intensity_perc = np.ma.masked_equal(drought_intensity_perc,999.0)
drought_lengths_perc = np.ma.masked_equal(drought_lengths_perc, 0)
drought_areas_perc = np.ma.masked_equal(spei_perc_object_area_m2, 0)
drought_severity_SVI = np.ma.masked_equal(SVI_drought_severity,999.0)

# reshape to a monthly array
nyears = int(ntime/12)
drought_severity_4d = drought_severity_perc.reshape((nyears,12,nlat,nlon))
drought_magnitude_4d = drought_magnitude_perc.reshape((nyears,12,nlat,nlon))
drought_intensity_4d = drought_intensity_perc.reshape((nyears,12,nlat,nlon))
drought_lengths_4d = drought_lengths_perc.reshape((nyears,12,nlat,nlon))
drought_onsets_4d = drought_onsets_perc.reshape((nyears,12,nlat,nlon))
drought_areas_4d = drought_areas_perc.reshape((nyears,12,nlat,nlon))
drought_severity_SVI_4d = drought_severity_SVI.reshape((nyears,12,nlat,nlon)) 


### Calculating annual frequency, DIMS, SVI severity, and area

For each pixel, find
  * annual drought area,
  * annual drought frequency,
  * annual SVI severity during droughts,
  * annual average drought intensity,
  * annual drought duration,
  * annual drought severity, and
  * annual drought magnitude.

In [ ]:
yearly_drought_frequency = np.ma.sum(drought_onsets_4d,axis=1)
yearly_drought_intensity = np.ma.mean(drought_intensity_4d,axis=1)
yearly_drought_duration = np.ma.mean(drought_lengths_4d, axis=1)
yearly_drought_severity = np.ma.mean(drought_severity_4d, axis=1)
yearly_drought_magnitude = np.ma.mean(drought_magnitude_4d, axis=1)
yearly_drought_area = np.ma.mean(drought_areas_4d, axis=1)
yearly_SVI_severity = np.ma.mean(drought_severity_SVI_4d, axis=1)  

For each pixel, find dispersion using metric from [Mailier et al. (2006)](https://journals.ametsoc.org/view/journals/mwre/134/8/mwr3160.1.xml)

In [ ]:
mean_drought_frequency = np.ma.mean(yearly_drought_frequency,axis=0)
var_drought_frequency = np.ma.var(yearly_drought_frequency,axis=0)
all_drought_dispersion = var_drought_frequency/mean_drought_frequency-1

### Calculate frequency, DIMS, dispersion, SVI severity, and area for selected periods. 

In [ ]:
period_year0 = [1995, 2000, 2005] #start year of period
period_year1 = [2009, 2009, 2009] #end year of period
nperiods = len(period_year0) # number of periods
years = np.unique(time_xr.dt.year) # find years in subsetted dataset

# create arrays to store period averages
period_drought_intensity = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_drought_duration = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_drought_frequency = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_drought_severity = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_drought_magnitude = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_drought_area = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_drought_dispersion = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_NDVI_mean = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))
period_SVI_severity = np.full(fill_value=-999.0,shape=(nperiods,nlat,nlon))

for pp in range(0,nperiods,1):
    period_year_inds = np.where((years>=period_year0[pp])&(years<=period_year1[pp]))[0]
    period_drought_intensity[pp,:,:] = np.ma.mean(yearly_drought_intensity[period_year_inds,:,:], axis=0)
    period_drought_duration[pp,:,:] = np.ma.mean(yearly_drought_duration[period_year_inds,:,:], axis=0)
    period_drought_frequency[pp,:,:] = np.ma.mean(yearly_drought_frequency[period_year_inds,:,:], axis=0)
    period_drought_severity[pp,:,:] = np.ma.mean(yearly_drought_severity[period_year_inds,:,:], axis=0)
    period_drought_magnitude[pp,:,:] = np.ma.mean(yearly_drought_magnitude[period_year_inds,:,:], axis=0)
    period_drought_area[pp,:,:] = np.ma.mean(yearly_drought_area[period_year_inds,:,:], axis=0)
    period_drought_frequency_var = np.ma.var(yearly_drought_frequency[period_year_inds,:,:], axis=0)
    period_drought_dispersion[pp,:,:] = period_drought_frequency_var/period_drought_frequency[pp,:,:] - 1
    period_NDVI_mean[pp,:,:] = np.ma.mean(NDVI_annual_regridded_np_padded[period_year_inds,:,:], axis=0)
    period_SVI_severity[pp,:,:] = np.ma.mean(yearly_SVI_severity[period_year_inds,:,:], axis=0)

### Nighttime Lights City Centers

Find closest population centers using processed night time lights. nighttime_lights.csv can be found in the GitHub.

In [ ]:
lights_df = pd.read_csv(dir_lights+'nighttime_lights.csv')

In [2]:
# This function allows us to find the closest city center location - will be used in loop at the end.
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return array[idx]

For each region and period, read in remittance data, merge with SPEI data, ensure NDVI is not NAN, find population centers, save output file.

In [ ]:
regions = ['nigeria','kenya','burkina faso','south africa', 'uganda','senegal']

# for "regridding" survey lat/longs to SPEI lat/longs
precision = 2
base = 0.05

# create SPEI data frame
lat_2d = np.repeat(np.around(lat, decimals=2),len(lon))
lon_2d = np.tile(np.around(lon, decimals=2),len(lat))

#select period
for pp in range(0,nperiods,1):
    print(str(period_year0[pp])+'-'+str(period_year1[pp]))
    spei_df = pd.DataFrame({
        'lat_SPEI':lat_2d,'long_SPEI':lon_2d,
        'intensity': period_drought_intensity[pp,:,:].flat,
        'magnitude': period_drought_magnitude[pp,:,:].flat,
        'duration': period_drought_duration[pp,:,:].flat,
        'frequency': period_drought_frequency[pp,:,:].flat,
        'severity': period_drought_severity[pp,:,:].flat,
        'dispersion': period_drought_dispersion[pp,:,:].flat,
        'area': period_drought_area[pp,:,:].flat,
        'NDVI_mean': period_NDVI_mean[pp,:,:].flat,
        'SVI_severity': period_SVI_severity[pp,:,:].flat,
        })
    for rr in regions:
        print(rr)
        remit_df = pd.read_csv(dir_rem+rr+'.csv', encoding = "ISO-8859-1")
        existing_columns = list(set(remit_df.columns) & set(spei_df.columns))
        remit_df = remit_df.drop(columns=existing_columns)
        if 'lat_regrid' in remit_df.columns:
            remit_df = remit_df.drop(columns=['lat_regrid','long_regrid'])
        remit_lat = np.array(remit_df['lat'])
        remit_lat_rounded = np.round(base*np.round(remit_lat/base), decimals=precision)
        remit_lon = np.array(remit_df['long'])
        remit_lon_rounded = np.round(base*np.round(remit_lon/base), decimals=precision)
        remit_df['lat_SPEI'] = remit_lat_rounded
        remit_df['long_SPEI'] = remit_lon_rounded
        remit_df['NDVI_lat'] = remit_df['lat_SPEI']
        remit_df['NDVI_lon'] = remit_df['long_SPEI']
        #city centers (nighttime lights)
        region_lights_df = lights_df[lights_df['NAME_0']==rr.title()].reset_index(drop=True)
        lights_lat = region_lights_df['latitude']
        lights_lon = region_lights_df['longitude']
        remit_lat_nearest = np.zeros(len(remit_lat))
        for ii in range(0,len(remit_lat),1):
            remit_lat_nearest[ii] = find_nearest(lights_lat,remit_lat[ii])
        remit_lon_nearest = np.zeros(len(remit_lon))
        for jj in range(0,len(remit_lon),1):
            remit_lon_nearest[jj] = find_nearest(lights_lon,remit_lon[jj]) 
        remit_df['pop_cent_lat'] = remit_lat_nearest
        remit_df['pop_cent_long'] = remit_lon_nearest
        #merge remittance dataframe with spei dataframe by matching regridded lat longs
        spei_remit_df = pd.merge(remit_df,spei_df,how='left',on=['lat_SPEI','long_SPEI'])
        rows_with_nan = spei_remit_df[(spei_remit_df['NDVI_mean'].isna())&~(spei_remit_df['lat'].isna())]
        if len(rows_with_nan)>0:
            period_NDVI_mean_xr = xr.DataArray(period_NDVI_mean[pp,:,:],coords = {'lat' : spei_sub_xr.lat, 'lon' : spei_sub_xr.lon}, name='NDVI_period')
            print(str(len(rows_with_nan)) + ' NDVI nans')
            # first try to fix using xarray find nearest using original lat long
            for nn in rows_with_nan.index:
                NDVI_missing = period_NDVI_mean_xr.sel(lat=rows_with_nan.loc[nn]['lat'], lon = rows_with_nan.loc[nn]['long'], method='nearest')
                if np.isnan(NDVI_missing.values.item()):
                # incrementally change lat and long until non-nan value is reached
                    for ii in np.arange(-10,11,1):
                        ii_check = period_NDVI_mean_xr.sel(lat=rows_with_nan.loc[nn]['lat']+base*ii, lon = rows_with_nan.loc[nn]['long'], method='nearest') 
                        if ~np.isnan(ii_check.values.item()):
                            spei_remit_df.at[nn,'NDVI_mean'] = ii_check.values.item()
                            spei_remit_df.at[nn,'NDVI_lat'] = ii_check.lat
                            spei_remit_df.at[nn,'NDVI_lon'] = ii_check.lon
                            break
                        else:
                            jj_check = period_NDVI_mean_xr.sel(lat=rows_with_nan.loc[nn]['lat']+base*ii, lon = rows_with_nan.loc[nn]['long']+base*ii, method='nearest')
                            if ~np.isnan(jj_check.values.item()):
                                spei_remit_df.at[nn,'NDVI_mean'] = jj_check.values.item()
                                spei_remit_df.at[nn,'NDVI_lat'] = jj_check.lat
                                spei_remit_df.at[nn,'NDVI_lon'] = jj_check.lon
                                break
                else:
                    spei_remit_df.at[nn,'NDVI_mean'] = NDVI_missing.values.item()
                    spei_remit_df.at[nn,'NDVI_lat'] = NDVI_missing.lat
                    spei_remit_df.at[nn,'NDVI_lon'] = NDVI_missing.lon  
        #save CSV file
        spei_remit_df.to_csv(dir_rem+rr.replace(" ","")+'_SPEI-'+str(spei_time_scales[tt])+'_'+str(period_year0[pp])+'-'+str(period_year1[pp])+'_'+str(percentiles[xx])+'percentile_'+str(years[0])+'-'+str(years[-1])+'_allcolumns.csv')
        spei_remit_stripped_df = spei_remit_df[['house', 'country', 'frequency', 'severity', 'dispersion', 'area', 'NDVI_mean', 'SVI_severity']]
        spei_remit_stripped_df.to_csv(dir_rem+rr.replace(" ","")+'_SPEI-'+str(spei_time_scales[tt])+'_'+str(period_year0[pp])+'-'+str(period_year1[pp])+'_'+str(percentiles[xx])+'percentile_'+str(years[0])+'-'+str(years[-1])+'.csv')
